# 01. Basic Tracing

In this notebook, you will learn how to connect Langfuse with Claude (Anthropic) and automatically trace LLM calls.

## What You Will Learn
- Initializing and verifying the Langfuse client connection
- Setting up auto-tracing with `AnthropicInstrumentor`
- Making Claude API calls and viewing traces in the Langfuse dashboard
- Streaming response tracing

## What is Langfuse?

**Langfuse** is an open-source platform that provides **observability** for LLM applications.

- Records the input/output, token usage, and latency of each LLM call
- Visualizes traces in a hierarchical structure
- Provides prompt management, evaluation, and experimentation features

```
Trace (a single request flow)
  └─ Span (an arbitrary unit of work)
       └─ Generation (an LLM call)
```

## Step 1: Install Packages

In [ ]:
%pip install langfuse anthropic opentelemetry-instrumentation-anthropic python-dotenv -q

## Step 2: Load Environment Variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Map CLAUDE_API_KEY from .env to ANTHROPIC_API_KEY used by the Anthropic SDK
os.environ["ANTHROPIC_API_KEY"] = os.environ["CLAUDE_API_KEY"]

print("Environment variables loaded")
print(f"Langfuse Base URL: {os.environ.get('LANGFUSE_BASE_URL')}")

## Step 3: Initialize Langfuse Client

`get_client()` automatically reads the keys from environment variables and initializes the Langfuse client.

In [ ]:
from langfuse import get_client

langfuse = get_client()

if langfuse.auth_check():
    print("✅ Langfuse auth successful! Connected to the dashboard.")
else:
    print("❌ Auth failed. Please check the keys in your .env file.")

## Step 4: Set Up Anthropic Auto-Tracing

Once you call `AnthropicInstrumentor().instrument()`,
all subsequent Anthropic SDK calls will be automatically recorded in Langfuse.

It is based on the OpenTelemetry (OTel) standard, enabling tracing without any code modifications.

In [ ]:
from opentelemetry.instrumentation.anthropic import AnthropicInstrumentor

AnthropicInstrumentor().instrument()

print("✅ Anthropic auto-tracing enabled")

## Step 5: First Claude API Call

Use a standard Anthropic client. Since `AnthropicInstrumentor` wraps it internally, no code changes are needed.

In [ ]:
from anthropic import Anthropic

client = Anthropic()

message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    messages=[
        {
            "role": "user",
            "content": "Why is LLM Observability important? Give 3 concise reasons."
        }
    ]
)

print(message.content[0].text)

# In short-lived scripts, call flush() to send buffered data to Langfuse
langfuse.flush()

> **Check the Langfuse Dashboard**
>
> [https://cloud.langfuse.com](https://cloud.langfuse.com) → Select your project → **Tracing** → **Traces**
>
> You should see the API call you just made recorded as a trace.
> Click on it to view detailed information including input, output, model, token count, and latency.

## Step 6: Call with System Prompt

In [ ]:
message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    system="You are a friendly cooking expert. Always respond in English.",
    messages=[
        {
            "role": "user",
            "content": "What are 3 key tips for making a delicious stew?"
        }
    ]
)

print(message.content[0].text)
langfuse.flush()

## Step 7: Streaming Response Tracing

Streaming calls are also automatically traced.

In [ ]:
print("Streaming response:\n")

with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=256,
    messages=[
        {
            "role": "user",
            "content": "Summarize the advantages of Python in one sentence."
        }
    ]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print("\n")
langfuse.flush()

## Step 8: Multi-Turn Conversation Tracing

In [ ]:
conversation_history = []

def chat(user_message: str) -> str:
    conversation_history.append({"role": "user", "content": user_message})
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system="You are a helpful AI assistant.",
        messages=conversation_history
    )
    
    assistant_message = response.content[0].text
    conversation_history.append({"role": "assistant", "content": assistant_message})
    
    return assistant_message


print("[Message 1]")
print(chat("Hello! My name is Alex."))
print()

print("[Message 2]")
print(chat("Do you remember my name?"))

langfuse.flush()

## Summary

What we covered in this notebook:

| Concept | Description |
|------|------|
| `get_client()` | Initialize the Langfuse client from environment variables |
| `auth_check()` | Verify connection to the Langfuse server |
| `AnthropicInstrumentor().instrument()` | Automatically trace all Anthropic calls |
| `langfuse.flush()` | Immediately send buffered data to Langfuse |

Next: **02_decorator_and_nesting.ipynb** → Fine-grained control with the `@observe()` decorator